# Fold batch norm

## Imports

In [ ]:
import torch
import torch.nn as nn

import numpy as np

from model.Model import MiniFCOSFaceV1
from inference.preprocess import preprocess_own_image
from dataset.Dataset import WiderFaceDataset

## Float values from CNN

In [ ]:
import numpy as np

weights = np.load("../model/export/minifcos_face_v1_weights.npz")

print(f"{'Layer name':45s} {'Shape':20s} {'Number of elements':>15s}")
total_params = 0
for name in weights.files:
    w = weights[name]
    n = w.size
    total_params += n
    print(f"{name:45s} {str(w.shape):20s} {n:15d}")

print(f"\nTotal parameters: {total_params}")

## Folding batch normalization

In [ ]:

eps = 1e-5  # default nn.BatchNorm2d eps

def fold_conv_bn(w_conv, gamma, beta, running_mean, running_var, eps=1e-5):
    scale = gamma / np.sqrt(running_var + eps)         
    w_fused = w_conv * scale[:, None, None, None]       
    b_fused = beta - running_mean * scale
    return w_fused.astype(np.float32), b_fused.astype(np.float32)

pairs = [
    ("stem.layers.0",   "stem.layers.1"),
    ("block1.layers.0", "block1.layers.1"),
    ("block2.layers.0", "block2.layers.1"),
    ("head.0.layers.0", "head.0.layers.1"),
]

folded = {}

for conv_prefix, bn_prefix in pairs:
    w_conv = weights[f"{conv_prefix}.weight"]
    gamma  = weights[f"{bn_prefix}.weight"]
    beta   = weights[f"{bn_prefix}.bias"]
    mean   = weights[f"{bn_prefix}.running_mean"]
    var    = weights[f"{bn_prefix}.running_var"]

    w_f, b_f = fold_conv_bn(w_conv, gamma, beta, mean, var, eps)

    layer_name = conv_prefix.rsplit(".", 1)[0] 
    folded[f"{layer_name}.weight"] = w_f
    folded[f"{layer_name}.bias"]   = b_f

folded["head.1.weight"] = weights["head.1.weight"]
folded["head.1.bias"]   = weights["head.1.bias"]

for name, w in folded.items():
    print(f"{name:20s} {str(w.shape):20s} {w.size}")

export_dir = Path("export")
export_dir.mkdir(exist_ok=True)
folded_path = export_dir / "minifcos_face_v1_folded_weights.npz"

np.savez_compressed(folded_path, **folded)

print("Folded weights saved:", folded_path)

### Checking folding

In [ ]:
n_before = sum(weights[n].size for n in weights.files if "num_batches_tracked" not in n)
n_after  = sum(w.size for w in folded.values())
print("Before BN folding:", n_before)
print("After BN folding:", n_after)
print("Difference:", n_before - n_after)

In [ ]:
#Load regular model
checkpoint_path = "../train/checkpoints/minifcos_face_v1_best.pt"
checkpoint = torch.load(checkpoint_path, map_location="cpu")
state_dict = checkpoint["model_state_dict"]

model_orig = MiniFCOSFaceV1()
model_orig.load_state_dict(state_dict)
model_orig.eval()

#Folded CNN
class ConvFolded(nn.Module):
    """Conv (s biasom) -> ReLU, bez BN — koristi se za foldane blokove."""
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1):
        super().__init__()
        padding = kernel_size // 2
        self.conv = nn.Conv2d(
            in_channels, out_channels, kernel_size,
            stride=stride, padding=padding, bias=True
        )
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.relu(self.conv(x))

class MiniFCOSFaceV1Folded(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem   = ConvFolded(3, 16, kernel_size=3, stride=2)
        self.block1 = ConvFolded(16, 24, kernel_size=3, stride=2)
        self.block2 = ConvFolded(24, 48, kernel_size=3, stride=2)
        self.head0  = ConvFolded(48, 48, kernel_size=3, stride=1)
        self.head1  = nn.Conv2d(48, 6, kernel_size=1, stride=1, padding=0)
    def forward(self, x):
        x = self.stem(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.head0(x)
        return self.head1(x)

#Load folded CNN
model_folded = MiniFCOSFaceV1Folded()

name_map = {
    "stem.layers.weight":   "stem.conv.weight",
    "stem.layers.bias":     "stem.conv.bias",
    "block1.layers.weight": "block1.conv.weight",
    "block1.layers.bias":   "block1.conv.bias",
    "block2.layers.weight": "block2.conv.weight",
    "block2.layers.bias":   "block2.conv.bias",
    "head.0.layers.weight": "head0.conv.weight",
    "head.0.layers.bias":   "head0.conv.bias",
    "head.1.weight":        "head1.weight",
    "head.1.bias":          "head1.bias",
}

folded_state_dict = {}
for old_name, new_name in name_map.items():
    folded_state_dict[new_name] = torch.from_numpy(folded[old_name])

model_folded.load_state_dict(folded_state_dict)
model_folded.eval()

torch.manual_seed(0)
dummy_input = torch.randn(1, 3, 320, 320)

with torch.no_grad():
    out_orig   = model_orig(dummy_input)
    out_folded = model_folded(dummy_input)

diff = (out_orig - out_folded).abs()
print("Output shape:", out_orig.shape)
print("Max abs diff:", diff.max().item())
print("Mean abs diff:", diff.mean().item())

## Float max/min values

In [ ]:
print(f"{'Layer':20s} {'Min':>10s} {'Max':>10s} {'Mean':>10s} {'Std':>10s}")
for name, w in folded.items():
    print(f"{name:20s} {w.min():10.4f} {w.max():10.4f} {w.mean():10.4f} {w.std():10.4f}")

## Activation min/max values

In [ ]:
validate_dataset = WiderFaceDataset('../dataset/WIDER_val/images', '../dataset/wider_face_split/wider_face_val_bbx_gt.txt')
validate_dataloader = DataLoader(validate_dataset)

activations = {}

def get_hook(name):
    def hook(module, input, output):
        activations[name] = output.detach()
    return hook

model_folded.stem.register_forward_hook(get_hook("stem"))
model_folded.block1.register_forward_hook(get_hook("block1"))
model_folded.block2.register_forward_hook(get_hook("block2"))
model_folded.head0.register_forward_hook(get_hook("head0"))
model_folded.head1.register_forward_hook(get_hook("head1"))

layer_names = ["stem", "block1", "block2", "head0", "head1"]

#Accumulators per layer
stats = {
    name: {
        "min": np.inf,
        "max": -np.inf,
        "sum": 0.0,
        "sum_sq": 0.0,
        "count": 0,
        "samples": []
    }
    for name in layer_names
}

N_IMAGES = 200        
SUBSAMPLE_STRIDE = 50

model_folded.eval()
with torch.no_grad():
    for idx, batch in enumerate(validate_dataloader):
        if N_IMAGES is not None and idx >= N_IMAGES:
            break

        img_tensor = batch[0] if isinstance(batch, (list, tuple)) else batch
        _ = model_folded(img_tensor)

        for name in layer_names:
            a = activations[name].numpy().ravel()

            s = stats[name]
            s["min"] = min(s["min"], a.min())
            s["max"] = max(s["max"], a.max())
            s["sum"] += a.sum()
            s["sum_sq"] += (a ** 2).sum()
            s["count"] += a.size
            s["samples"].append(a[::SUBSAMPLE_STRIDE])

        if idx % 20 == 0:
            print(f"Processed {idx} pictures")

print(f"\n{'Layer':10s} {'Min':>10s} {'Max':>10s} {'Mean':>10s} {'Std':>10s} {'P99.9':>10s}")
for name in layer_names:
    s = stats[name]
    mean = s["sum"] / s["count"]
    var = s["sum_sq"] / s["count"] - mean ** 2
    std = np.sqrt(max(var, 0))
    all_samples = np.concatenate(s["samples"])
    p999 = np.percentile(all_samples, 99.9)
    print(f"{name:10s} {s['min']:10.4f} {s['max']:10.4f} {mean:10.4f} {std:10.4f} {p999:10.4f}")